# Bölüm 17: Dağıtım Seçenekleri
**İlk LLM'inizi Oluşturun — Bölüm 17: Dağıtım Seçenekleri**

Bu notebook, Bölüm 17'deki çalıştırılabilir kod örneklerini bir araya getirir. Modal dağıtımı, Colab değil, terminalinizden çalışır, ancak bu notebook her adımı açıklar.

- Kurulumlar: modal
- Veri: satır içi örnekler; harici dosyalara ihtiyaç yok
- Çalışma zamanı: Çoğu Modal komutu notebook değil, terminalde çalışır

In [ ]:
# ===== KURULUM =====
# Modal'ı kur
!pip install -q modal

print('Modal kuruldu!')
print('ÖNEMLİ: Kimlik doğrulama için terminalinizde "modal setup" komutunu çalıştırın.')
print('Bu, giriş için bir tarayıcı penceresi açar.')

## Bölüm 17.1: Neden Modal?

**Sunucusuz (Serverless) Nedir?**

| Geleneksel Sunucu | Sunucusuz (Modal) |
|-------------------|-------------------|
| 7/24 bir bilgisayar kiralarsınız | Kod yalnızca gerektiğinde çalışır |
| Boştayken bile ödeme yaparsınız | Hesaplama saniyesi başına ödeme yaparsınız |
| Güncellemeleri, güvenliği siz yönetirsiniz | Platform her şeyi halleder |
| Sabit kapasite | Otomatik ölçekleme |

**Analoji:** Geleneksel barındırma, araba sahibi olmak gibidir. Sunucusuz, taksi kullanmak gibidir; sadece yolculuk yaptığınızda ödersiniz.

**LLM'ler için Neden Modal?**
- Yerleşik GPU desteği (T4'ten H100'e)
- Ayda 30$ ücretsiz kredi (~50 saat T4 GPU süresi)
- Python'a özgü (Docker yok, YAML yok)
- Sıfıra ölçekler (boştayken ücret yok)

## Bölüm 17.2: Merhaba Modal

En basit Modal uygulaması. Bunu `hello.py` olarak kaydedin ve `modal run hello.py` ile çalıştırın:

In [ ]:
# Bunu hello.py olarak kaydedin
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Hello, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Bu kodu 'hello.py' olarak kaydedin:")
print(hello_modal_code)
print("\nArdından çalıştırın: modal run hello.py")

**Ne oldu?**

1. `modal.App()` uygulamanızı oluşturur
2. `@app.function()` Modal'ın bulutunda çalışacak kodu işaretler
3. `hello.remote()` fonksiyonu uzaktan çağırır
4. Modal bir container başlattı, kodunuzu çalıştırdı, sonucu döndürdü

## Bölüm 17.3: GPU Desteği Ekleme

Tek bir parametre GPU desteği ekler:

In [ ]:
# Bunu gpu_test.py olarak kaydedin
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

@app.function(gpu="T4")  # İşte bu kadar!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Bu kodu 'gpu_test.py' olarak kaydedin:")
print(gpu_test_code)
print("\nArdından çalıştırın: modal run gpu_test.py")
print("\nBeklenen çıktı: GPU available: Tesla T4")

**Tek bir parametre.** CUDA kurulumu yok, sürücü yönetimi yok, Docker imajları yok. Modal tüm GPU yığınını halleder.

## Bölüm 17.4: Bağımlılıkları Yükleme

LLM'iniz kütüphanelere ihtiyaç duyar. Modal, container ortamınızı Python'da tanımlamanıza izin verir:

In [ ]:
# Bunu with_deps.py olarak kaydedin
with_deps_code = '''
import modal

# Container imajını tanımla
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Bu kodu 'with_deps.py' olarak kaydedin:")
print(with_deps_code)
print("\nArdından çalıştırın: modal run with_deps.py")

`image` parametresi Modal'a neyin yükleneceğini söyler. İlk çalıştırma daha uzun sürer (imaj oluşturulurken), ancak sonraki çalıştırmalar önbelleğe alınmış imajı yeniden kullanır.

## Bölüm 17.5: FastAPI ile Web Uç Noktaları

Herkesin erişebileceği bir web uç noktası oluşturun. Bölüm 16'daki FastAPI bilgisinin işe yaradığı yer burası:

In [ ]:
# Bunu simple_api.py olarak kaydedin
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Hello, {name}!"}

@app.function()
@modal.asgi_app()
def serve():
    return web_app
'''

print("Bu kodu 'simple_api.py' olarak kaydedin:")
print(simple_api_code)
print("\nArdından çalıştırın: modal deploy simple_api.py")
print("\nŞu şekilde bir URL alacaksınız: https://your-workspace--my-api-serve.modal.run")

**API'niz yayında!** O URL'yi tarayıcınızda açın. Yola `/health` ekleyin:

```
https://your-workspace--my-api-serve.modal.run/health
```

Şunu görmelisiniz: `{"status": "healthy"}`

## Bölüm 17.6: Eksiksiz LLM Servisi

İşte üretime hazır bir LLM dağıtımı:

In [ ]:
# Bunu llm_service.py olarak kaydedin
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os

# LLM bağımlılıklarıyla container imajını tanımla
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.6.4",  # Tekrarlanabilirlik için sürümü sabitle
    "torch",
)

app = modal.App("my-llm-service")
web_app = FastAPI(title="My LLM API", version="1.0.0")

# İstek/Yanıt modelleri (Bölüm 16'dan tanıdık)
class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

class ChatResponse(BaseModel):
    response: str

# Global model referansı (container başına bir kez yüklenir)
_model = None

def get_model():
    """Modeli bir kez yükle, tüm istekler için yeniden kullan."""
    global _model
    if _model is None:
        from vllm import LLM
        _model = LLM(
            model="Qwen/Qwen2.5-1.5B-Instruct",
            trust_remote_code=True,
        )
    return _model

@web_app.get("/health")
def health():
    """Sağlık kontrolü uç noktası (Bölüm 16 hatırlatması)."""
    return {"status": "healthy", "model": "Qwen2.5-1.5B-Instruct"}

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    """Bir mesaja yanıt üret."""
    try:
        model = get_model()
        from vllm import SamplingParams
        
        params = SamplingParams(max_tokens=request.max_tokens)
        outputs = model.generate([request.message], params)
        response_text = outputs[0].outputs[0].text
        
        return ChatResponse(response=response_text)
    except Exception as e:
        raise HTTPException(500, f"Generation failed: {str(e)}")

@app.function(
    image=image,
    gpu="T4",
    timeout=300,
    scaledown_window=300,  # 5 dakika sıcak tut
)
@modal.concurrent(max_inputs=10)  # Container başına birden fazla isteği işle
@modal.asgi_app()
def serve():
    return web_app
'''

print("Bu kodu 'llm_service.py' olarak kaydedin:")
print(llm_service_code)

In [ ]:
print("Şununla dağıtın: modal deploy llm_service.py")
print("\nİlk dağıtım birkaç dakika sürer (model indiriliyor).")
print("Sonraki dağıtımlar hızlıdır.")
print("\nŞununla test edin:")
print('curl https://your-url.modal.run/health')
print('curl -X POST https://your-url.modal.run/chat -H "Content-Type: application/json" -d \'{"message": "What is Python?"}\'')  

## Bölüm 17.7: Sırlar ve Hacimler

### Sırlar Oluşturma

```bash
# Komut satırından bir sır oluştur
modal secret create my-secrets API_KEY=your_secret_key
```

### Kodda Sırları Kullanma

In [ ]:
# Kodunuzda sırları kullanma
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Anahtarı güvenli bir şekilde kullan...
'''

print("Sırları kullanma:")
print(secrets_example)

In [ ]:
# Modelleri önbelleğe almak için hacimleri kullanma
volumes_example = '''
# Model önbelleği için bir hacim oluştur
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="T4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Modeller bir kez indirilir, sonra hacimde önbelleğe alınır
    pass
'''

print("Modelleri önbelleğe almak için hacimleri kullanma:")
print(volumes_example)
print("\nİlk istek modeli indirir. Sonraki istekler önbelleği kullanır.")

## Bölüm 17.8: GPU Seçenekleri

| Model Boyutu | Önerilen GPU | Maliyet/Saat |
|--------------|--------------|-------------|
| < 3B parametre | T4 (16GB) | $0.59 |
| 3-8B parametre | A10G (24GB) | $1.10 |
| 8-30B parametre | A100-40GB | $2.50 |
| 30B+ parametre | A100-80GB / H100 | $4-8 |

**T4 ile başlayın.** Yalnızca gerektiğinde yükseltin.

In [ ]:
# GPU seçimi örnekleri
print("GPU seçimi tek bir parametredir:")
print()
print('@app.function(gpu="T4")      # Bütçe seçeneği')
print('@app.function(gpu="A10G")    # Orta düzey')
print('@app.function(gpu="A100")    # Yüksek performans')
print('@app.function(gpu="H100")    # Maksimum güç')
print('@app.function(gpu="A100:2")  # İki A100')

## Bölüm 17.9: Dağıtımınızı Yönetme

### Logları Görüntüleme

```bash
modal app logs my-llm-service
```

Ya da [modal.com](https://modal.com) adresindeki kontrol panelini kullanın.

### Uygulamanızı Güncelleme

```bash
# Sadece yeniden dağıt
modal deploy llm_service.py
```

Kesinti olmaz. Modal, sürekli güncellemeleri otomatik olarak halleder.

### Maliyet Kontrolü

- Boştaki container'ları kapatmak için `scaledown_window` kullanın
- T4 ile başlayın, yalnızca gerektiğinde yükseltin
- Harcamaları Modal kontrol panelinde izleyin

## Özet

Şunları öğrendiniz:

1. **Fonksiyonları dağıtmak** `@app.function()` ile Modal'a
2. **GPU desteği eklemek** tek bir parametreyle: `gpu="T4"`
3. **Bağımlılıkları yüklemek** `modal.Image` kullanarak
4. **Web uç noktaları oluşturmak** FastAPI + `@modal.asgi_app()` ile
5. **Sırları saklamak** güvenli bir şekilde `modal.Secret` ile
6. **Modelleri önbelleğe almak** `modal.Volume` ile
7. **Maliyetleri kontrol etmek** boşta kalma zaman aşımları ve izleme ile

**LLM'iniz artık dizüstü bilgisayarınızda hapsolmuş değil.** GPU hızlandırması ile herkese, her yerde erişilebilir.

**Hatırlanması gereken komutlar:**
```bash
pip install modal      # Kur
modal setup            # Kimlik doğrula
modal run file.py      # Bir kez çalıştır
modal deploy file.py   # Kalıcı olarak dağıt
modal app logs name    # Logları görüntüle
```